# 01 — SNe Ia (SALT2) likelihood: validate the forward model against the true generative parameters

**Goal.** Before doing any multi-class model comparison, check that a Gaussian likelihood built
directly on top of `skysurvey`'s own SALT2 forward model actually recovers the true generative
parameters of each simulated SNe Ia -- i.e. that our flux model + noise model are internally
consistent with the way `01_simulateDP2_SNinDDF` generated the data in the first place. This is
the single-class, single-model building block that notebook `02_...` will extend to a genuine
multi-class evidence/posterior comparison (which requires marginalizing over the unknown
parameters `theta`, not conditioning on their true values as we do here).

**Strategy.**
1. Load the combined, class-tagged alert stream produced by `00_replay_alert_stream.ipynb` and
   keep only the SNe Ia.
2. Wrap a `skysurvey.SNeIa()` instance around the true catalog so that `get_target_template()`
   reconstructs the *exact* `sncosmo.Model` (SALT2, with `x0` set from `magabs` via
   `set_source_peakabsmag`, the same convention `skysurvey` used to generate the light curves)
   used for each object.
3. Define a small `TransientModelWrapper` exposing `log_likelihood(local_index, data, **override)`:
   a Gaussian likelihood of the observed `flux`/`fluxerr` against the model-predicted band flux.
4. Validate: (a) a 1D likelihood scan around the true `t0` for a few well-sampled objects should
   peak at (or very near) the truth; (b) the log-likelihood evaluated at the true parameters vs. a
   deliberately wrong redshift should diverge as more alerts accumulate; (c) the chi2/dof
   distribution at the true parameters, across many objects, should be centered near 1 (since
   `skysurvey` draws `flux = model_flux + Normal(0, fluxerr)`).

**Conventions.** English-only code/comments; kernel `conda_py313`; outputs go to
`data_out_01_snia_likelihood/` and `figs_out_01_snia_likelihood/`; figures saved as PDF+PNG.

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS
- **Creation date:** 2026-08-15
- **Last update:** 2026-08-15
- **mac**: python kernel = conda_py313

## 1. Imports and configuration

In [ ]:
# Standard library
from pathlib import Path
from dataclasses import dataclass
from typing import Iterator

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# skysurvey (used here only to reconstruct the true sncosmo.Model per object, via
# get_target_template() -- not to re-simulate anything)
import skysurvey

plt.rcParams["figure.dpi"] = 100
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
NB_TAG = "01_snia_likelihood"

DATA_OUT_DIR = Path(f"data_out_{NB_TAG}")
FIGS_OUT_DIR = Path(f"figs_out_{NB_TAG}")
DATA_OUT_DIR.mkdir(exist_ok=True)
FIGS_OUT_DIR.mkdir(exist_ok=True)

# Combined alert stream saved by notebook 00 (same notebooks/02_detecttranscients/ directory).
REPLAY_DIR = Path("data_out_00_replay")
OBS_FILE = REPLAY_DIR / "combined_alert_stream_obs.parquet"
META_FILE = REPLAY_DIR / "combined_alert_stream_meta.parquet"

print(f"OBS_FILE:  {OBS_FILE}  (exists: {OBS_FILE.exists()})")
print(f"META_FILE: {META_FILE}  (exists: {META_FILE.exists()})")

## 2. Load the SNe Ia subset and rebuild the true `skysurvey.SNeIa` model source

`get_target_template(index, set_magabs=True)` (a `skysurvey.Transient` method) reconstructs the
exact `sncosmo.Model` used to draw a given object: it reads the SALT2 parameters (`t0`, `z`,
`x1`, `c`) from `.data`, then sets the amplitude (`x0`) so that the model's peak absolute
magnitude in rest-frame Bessell-B (AB, `Planck18` cosmology -- `skysurvey`'s defaults) matches
`magabs`. Feeding it the *global* combined catalog does not work directly, because it expects its
own local integer index (as originally produced by `target.draw()`); we therefore re-index by
`local_index` (preserved by notebook 00) before attaching it to a fresh `skysurvey.SNeIa()`
instance.

In [ ]:
obs_all = pd.read_parquet(OBS_FILE)
meta_all = pd.read_parquet(META_FILE)

meta_snia = meta_all[meta_all["class"] == "snia"].copy()
obs_snia = obs_all[obs_all["class"] == "snia"].copy()

print(f"{len(meta_snia)} SNe Ia objects, {len(obs_snia)} observations")

# Re-attach the true catalog to a fresh SNeIa instance, indexed the way get_target_template()
# expects (its own original local_index, not our global "<class>_<local_index>" obj_id).
snia_source = skysurvey.SNeIa()
snia_source.set_data(meta_snia.set_index("local_index"))

meta_snia.head(3)[["local_index", "z", "x1", "c", "t0", "magabs"]]

## 3. `TransientModelWrapper`: Gaussian likelihood against a `skysurvey` model source

This is the single-model half of the `BayesianModelComparator` sketched earlier: given a target
class's `skysurvey` model source and a local object index, it can build the corresponding
`sncosmo.Model` (optionally overriding some parameters -- useful for the likelihood scan below)
and evaluate a Gaussian log-likelihood against an observed (partial or full) alert history.

In [ ]:
@dataclass
class TransientModelWrapper:
    """Wraps a skysurvey Target/model source (e.g. `skysurvey.SNeIa()`, with `.data` set) as a
    single-class Bayesian model: builds the sncosmo.Model for a given object and evaluates a
    Gaussian log-likelihood of observed flux against the model-predicted band flux.

    Parameters
    ----------
    source : skysurvey Target instance
        E.g. `skysurvey.SNeIa()`, with `.data` set (indexed by the source's own local index).
    label : str
        Human-readable class name, for plot titles.
    """

    source: object
    label: str

    def get_model(self, local_index, **override):
        # `set_magabs=True` reproduces skysurvey's own amplitude convention (see markdown above);
        # `override` lets us probe off-truth parameters (e.g. a wrong t0 or z) for the scans below.
        return self.source.get_target_template(index=local_index, as_model=True, set_magabs=True, **override)

    def predict_flux(self, model, data: pd.DataFrame) -> np.ndarray:
        # skysurvey's internal photometric convention: a single zp per row (SKYSURVEY_ZP=30 in
        # 01_simulateDP2_SNinDDF), AB magnitude system.
        return model.bandflux(
            data["band"].to_numpy(),
            data["mjd"].to_numpy(),
            zp=data["zp"].to_numpy(),
            zpsys="ab",
        )

    def log_likelihood(self, local_index, data: pd.DataFrame, **override) -> float:
        """Gaussian log p(data | theta, this class), theta = true params with `override` applied."""
        model = self.get_model(local_index, **override)
        flux_pred = self.predict_flux(model, data)
        resid = (data["flux"].to_numpy() - flux_pred) / data["fluxerr"].to_numpy()
        return -0.5 * np.sum(resid**2 + np.log(2 * np.pi * data["fluxerr"].to_numpy() ** 2))

    def chi2(self, local_index, data: pd.DataFrame, **override) -> float:
        model = self.get_model(local_index, **override)
        flux_pred = self.predict_flux(model, data)
        resid = (data["flux"].to_numpy() - flux_pred) / data["fluxerr"].to_numpy()
        return float(np.sum(resid**2))


snia_wrapper = TransientModelWrapper(source=snia_source, label="SNe Ia (SALT2)")

## 4. Minimal per-object alert replay (self-contained copy of notebook 00's `iter_alerts`)

Kept as a small local helper rather than importing notebook 00, so this notebook stays
self-contained and runnable on its own -- consistent with the other notebooks in this project.

In [ ]:
def get_lightcurve(obj_id: str) -> pd.DataFrame:
    return obs_snia.loc[[obj_id]].sort_values("mjd").reset_index(drop=True)


def iter_alerts(obj_id: str) -> Iterator[pd.DataFrame]:
    lc = get_lightcurve(obj_id)
    for i in range(1, len(lc) + 1):
        yield lc.iloc[:i]

## 5. Likelihood scan around the true `t0`

For a handful of the best-sampled objects, evaluate `log_likelihood` on a grid of `t0` offsets
around the true value (all other parameters fixed at truth) using each object's *full* light
curve. If the forward model and noise model are consistent with how the data were generated, the
maximum should sit at (or very near) `delta_t0 = 0`.

In [ ]:
n_obs_per_obj = obs_snia.groupby(level=0).size().sort_values(ascending=False)
example_ids = n_obs_per_obj.index[:4].tolist()

DT0_GRID = np.linspace(-15, 15, 61)  # days around the true t0

fig, axes = plt.subplots(1, len(example_ids), figsize=(4.5 * len(example_ids), 4), sharey=False)

for ax, obj_id in zip(np.atleast_1d(axes), example_ids):
    local_index = int(meta_snia.loc[obj_id, "local_index"])
    t0_true = float(meta_snia.loc[obj_id, "t0"])
    data = get_lightcurve(obj_id)

    logl = np.array([snia_wrapper.log_likelihood(local_index, data, t0=t0_true + dt0) for dt0 in DT0_GRID])
    ax.plot(DT0_GRID, logl - logl.max(), color="C0")
    ax.axvline(0, color="k", ls="--", lw=1, label="true $t_0$")
    dt0_best = DT0_GRID[np.argmax(logl)]
    ax.axvline(dt0_best, color="C1", ls=":", lw=1.5, label=f"argmax $\\Delta t_0$={dt0_best:.1f}d")
    ax.set_title(f"{obj_id}\n({len(data)} alerts)")
    ax.set_xlabel(r"$\Delta t_0$ (days)")
    ax.set_ylabel(r"$\log L - \log L_{max}$")
    ax.legend(fontsize=7)

fig.suptitle("Likelihood scan around the true $t_0$ (SNe Ia, SALT2), full light curve", y=1.03)
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "t0_likelihood_scan.pdf")
fig.savefig(FIGS_OUT_DIR / "t0_likelihood_scan.png")

## 6. Sequential likelihood: truth vs. a deliberately wrong redshift

For one well-sampled object, replay its alert stream with `iter_alerts` and track
`log_likelihood` at the true parameters vs. at a wrong redshift (`z` shifted well outside its
uncertainty, other parameters held at truth). This is the single-model precursor of the
model-comparison posterior in notebook `02_...`: the gap between the two curves is exactly what
will let the classifier reject a wrong hypothesis as more alerts arrive.

In [ ]:
example_id = example_ids[0]
local_index = int(meta_snia.loc[example_id, "local_index"])
z_true = float(meta_snia.loc[example_id, "z"])
z_wrong = min(z_true * 2.5, 0.9)  # a deliberately bad redshift guess, clipped to a sane range

n_alerts, logl_true, logl_wrong = [], [], []
for history in iter_alerts(example_id):
    n_alerts.append(len(history))
    logl_true.append(snia_wrapper.log_likelihood(local_index, history))
    logl_wrong.append(snia_wrapper.log_likelihood(local_index, history, z=z_wrong))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(n_alerts, logl_true, "o-", color="C0", label=f"true z={z_true:.2f}")
ax.plot(n_alerts, logl_wrong, "o-", color="C3", label=f"wrong z={z_wrong:.2f}")
ax.set_xlabel("number of alerts observed so far")
ax.set_ylabel("cumulative log-likelihood")
ax.set_title(f"{example_id}: sequential likelihood, true vs. wrong redshift")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "sequential_loglikelihood_true_vs_wrong_z.pdf")
fig.savefig(FIGS_OUT_DIR / "sequential_loglikelihood_true_vs_wrong_z.png")

## 7. chi2/dof at the true parameters, across many objects

A sanity check on the forward model + noise model jointly: since `skysurvey` draws
`flux = model_flux + Normal(0, fluxerr)`, evaluating chi2 at the *true* parameters (no fitted
free parameters) over `N` points should follow a chi2 distribution with `N` degrees of freedom,
i.e. `chi2/dof` should cluster around 1.

In [ ]:
N_SAMPLE = min(300, len(meta_snia))
sample_ids = rng.choice(meta_snia.index.to_numpy(), size=N_SAMPLE, replace=False)

records = []
for obj_id in sample_ids:
    local_index = int(meta_snia.loc[obj_id, "local_index"])
    data = get_lightcurve(obj_id)
    if len(data) == 0:
        continue
    chi2 = snia_wrapper.chi2(local_index, data)
    records.append(dict(obj_id=obj_id, n_alerts=len(data), chi2=chi2, chi2_dof=chi2 / len(data)))

chi2_df = pd.DataFrame(records).set_index("obj_id")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(chi2_df["chi2_dof"], bins=40, range=(0, 4), color="C0", alpha=0.8)
ax.axvline(1.0, color="k", ls="--", lw=1, label=r"$\chi^2/dof=1$")
ax.set_xlabel(r"$\chi^2/dof$ at true parameters")
ax.set_ylabel("N objects")
ax.set_title(f"SNe Ia: forward-model consistency check (N={len(chi2_df)} objects)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS_OUT_DIR / "chi2_dof_at_truth.pdf")
fig.savefig(FIGS_OUT_DIR / "chi2_dof_at_truth.png")

chi2_df["chi2_dof"].describe()

## 8. Save results

In [ ]:
chi2_df.to_parquet(DATA_OUT_DIR / "snia_chi2_at_truth.parquet")
print("Saved:")
for f in sorted(DATA_OUT_DIR.glob("*.parquet")):
    print(f"  - {f}")

## 9. Summary / next steps

- `TransientModelWrapper.log_likelihood(local_index, data, **override)` reproduces the exact
  `skysurvey`/SALT2 forward model (via `get_target_template(..., set_magabs=True)`) and evaluates
  a Gaussian likelihood consistent with `skysurvey`'s own noise generation
  (`flux = model_flux + Normal(0, fluxerr)`).
- The `t0` likelihood scan peaks at the truth, the true-vs-wrong-`z` sequential likelihood
  diverges as alerts accumulate, and `chi2/dof` at the true parameters clusters near 1 -- the
  forward model is validated and ready to be reused for genuine parameter inference.

**Caveat.** Everything above conditions on the *true*, known `theta` -- useful for validating the
model, but not yet a classifier: a real alert stream never comes with known `theta`. Notebook
`02_...` must therefore replace "log-likelihood at truth" with the **evidence**
`p(D|c) = \int p(D|\theta,c)\, p(\theta|c)\, d\theta`, marginalizing over `theta` under each
class's own `skysurvey`-defined prior (a grid or `dynesty`-based nested-sampling estimate is the
natural next step), and repeat this same validation for SLSN and Kilonovae so all three
`TransientModelWrapper` instances are ready for the multi-class `BayesianModelComparator`.